SHRIHARIHARAN S

24MCS1058

In [2]:
!pip install rank_bm25 transformers datasets #installing three Libraries

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 11.0 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2024.12.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is 

In [3]:
#Importing Libraries
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset
import numpy as np

In [4]:
#5 medical reports are in a list called 'reports'
#Data Preparation
reports = [
 "Patient presented with fever and cough for the past three days. The cough is primarily dry, and the patient reports occasional shortness of breath. No significant weight loss has been noted.",  # Report 1
 "Medical history includes hypertension controlled with medication, a previous episode of pneumonia, and seasonal allergies. The patient takes lisinopril and loratadine regularly. No known drug allergies.",  # Report 2
  "Physical examination revealed a temperature of 102°F, elevated heart rate of 110 bpm, and clear lungs upon auscultation. Mild wheezing was noted during forced expiration. The patient also exhibited mild dehydration.",  # Report 3
   "Lab results showed elevated white blood cell count (WBC 15,000/mm³), with a predominance of neutrophils. Chest X-ray findings were consistent with bronchitis, and no signs of consolidation or pleural effusion were noted.",           # Report 4
    "Treatment plan includes antibiotics such as azithromycin for the suspected bacterial infection, along with a prescription for a cough suppressant and instructions for increased fluid intake. The patient is advised to follow up in one week or sooner if symptoms worsen.",    # Report 5
]

In [5]:
tokenized_reports = [report.split() for report in reports] #, breaking reports into individual words.

bm25 = BM25Okapi(tokenized_reports) #creates a BM25 index from the tokenized reports

In [6]:
tokenized_reports

[['Patient',
  'presented',
  'with',
  'fever',
  'and',
  'cough',
  'for',
  'the',
  'past',
  'three',
  'days.',
  'The',
  'cough',
  'is',
  'primarily',
  'dry,',
  'and',
  'the',
  'patient',
  'reports',
  'occasional',
  'shortness',
  'of',
  'breath.',
  'No',
  'significant',
  'weight',
  'loss',
  'has',
  'been',
  'noted.'],
 ['Medical',
  'history',
  'includes',
  'hypertension',
  'controlled',
  'with',
  'medication,',
  'a',
  'previous',
  'episode',
  'of',
  'pneumonia,',
  'and',
  'seasonal',
  'allergies.',
  'The',
  'patient',
  'takes',
  'lisinopril',
  'and',
  'loratadine',
  'regularly.',
  'No',
  'known',
  'drug',
  'allergies.'],
 ['Physical',
  'examination',
  'revealed',
  'a',
  'temperature',
  'of',
  '102°F,',
  'elevated',
  'heart',
  'rate',
  'of',
  '110',
  'bpm,',
  'and',
  'clear',
  'lungs',
  'upon',
  'auscultation.',
  'Mild',
  'wheezing',
  'was',
  'noted',
  'during',
  'forced',
  'expiration.',
  'The',
  'patient',
 

In [7]:
bm25

In [8]:
model_name = "google/flan-t5-large"#loading the Large Language Models
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [9]:
#Defining the Retrieval Function
def retrieve_bm25(query, top_k=3):
    tokenized_query = query.split()
    doc_scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(doc_scores)[::-1][:top_k]
    return [reports[i] for i in top_indices]

def retrieve_random(query, top_k=1):
    # Randomly select k reports
    indices = np.random.choice(len(reports), size=top_k, replace=False)
    return [reports[i] for i in indices]

In [10]:
#defining the Querying Function
def query_llm(query, retrieved_docs):
    context = "\n".join(retrieved_docs)
    prompt = f"Context: {context}\nQuestion: {query}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer

In [11]:
#Queries
query = "What are the patient's symptoms?"

# BM25 Retrieval
bm25_docs = retrieve_bm25(query)
bm25_answer = query_llm(query, bm25_docs)

# Random Retrieval
random_docs = retrieve_random(query)
random_answer = query_llm(query, random_docs)

print("BM25 Answer:", bm25_answer)
print("Random Answer:", random_answer)


BM25 Answer: fever and cough
Random Answer: cough suppressant


In [12]:
query = "What are the patient's body Temperature?"

# BM25 Retrieval
bm25_docs = retrieve_bm25(query)
bm25_answer = query_llm(query, bm25_docs)

# Random Retrieval
random_docs = retrieve_random(query)
random_answer = query_llm(query, random_docs)

print("BM25 Answer:", bm25_answer)
print("Random Answer:", random_answer)


BM25 Answer: Fever
Random Answer: 102°F


In [13]:
query = "What are the patient's Treatment plan?"

# BM25 Retrieval
bm25_docs = retrieve_bm25(query)
bm25_answer = query_llm(query, bm25_docs)

# Random Retrieval
random_docs = retrieve_random(query)
random_answer = query_llm(query, random_docs)

print("BM25 Answer:", bm25_answer)
print("Random Answer:", random_answer)


BM25 Answer: antibiotics such as azithromycin
Random Answer: iv.


In [14]:
query = "What does the patient's lab report denotes?"

# BM25 Retrieval
bm25_docs = retrieve_bm25(query)
bm25_answer = query_llm(query, bm25_docs)

# Random Retrieval
random_docs = retrieve_random(query)
random_answer = query_llm(query, random_docs)

print("BM25 Answer:", bm25_answer)
print("Random Answer:", random_answer)

BM25 Answer: elevated white blood cell count
Random Answer: (iv)


In [15]:
query = "Generate the detailed report in 100 words"

# BM25 Retrieval
bm25_docs = retrieve_bm25(query)
bm25_answer = query_llm(query, bm25_docs)

# Random Retrieval
random_docs = retrieve_random(query)
random_answer = query_llm(query, random_docs)

print("BM25 Answer:", bm25_answer)
print("Random Answer:", random_answer)

BM25 Answer: The patient presents with fever and cough for the past three days. The cough is primarily dry,
Random Answer: The patient has a history of hypertension, which is controlled with medication. The patient has 


# Evaluation

In [16]:
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

In [17]:
def retrieve_bm25(query, top_k=3):
    tokenized_query = query.split()
    doc_scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(doc_scores)[::-1][:top_k]
    return top_indices

In [18]:
def retrieve_random(query, top_k=3):
    indices = np.random.choice(len(reports), size=top_k, replace=False)
    return indices

In [19]:
def evaluate_retrieval(queries, ground_truth, retrieval_function, top_k=3):
    precisions = []
    recalls = []
    f1_scores = []

    for query in queries:
        retrieved_indices = retrieval_function(query, top_k)
        relevant_indices = ground_truth[query]

        # Convert to binary vectors for evaluation
        retrieved_vector = [1 if i in retrieved_indices else 0 for i in range(len(reports))]
        relevant_vector = [1 if i in relevant_indices else 0 for i in range(len(reports))]

        precisions.append(precision_score(relevant_vector, retrieved_vector))
        recalls.append(recall_score(relevant_vector, retrieved_vector))
        f1_scores.append(f1_score(relevant_vector, retrieved_vector))

    return np.mean(precisions), np.mean(recalls), np.mean(f1_scores)


In [20]:
tokenized_reports = [report.split() for report in reports]  # Tokenize reports
bm25 = BM25Okapi(tokenized_reports)

In [21]:
ground_truth = {
    "What are the patient's symptoms?": [0,2],
    "What are the patient's body Temperature?": [2],
    "What are the patient's Treatment plan?": [4],
    "What does the patient's lab report denotes?": [3],
    "Generate the detailed report in 100 words": [0,1,2,3,4],
}
queries = list(ground_truth.keys())

In [22]:
bm25_precision, bm25_recall, bm25_f1 = evaluate_retrieval(queries, ground_truth, retrieve_bm25)
print(f"BM25: Precision={bm25_precision:.2f}, Recall={bm25_recall:.2f}, F1={bm25_f1:.2f}")

BM25: Precision=0.40, Recall=0.62, F1=0.43


In [23]:
random_precision, random_recall, random_f1 = evaluate_retrieval(queries, ground_truth, retrieve_random)
print(f"Random: Precision={random_precision:.2f}, Recall={random_recall:.2f}, F1={random_f1:.2f}")

Random: Precision=0.47, Recall=0.82, F1=0.53


Inference:

Random Retrieval Performed Better Overall: Despite its simplicity, random selection achieved higher recall and F1-score, suggesting it captured more relevant information by chance.

BM25 Precision Could Be Improved: BM25 had lower precision, indicating room for improvement in how it identifies relevant reports. Parameter tuning or a different retrieval approach might be needed.

Recall is Crucial in This Context: The higher recall of random selection likely contributed to its better performance. In medical question answering, retrieving as many relevant reports as possible is often preferred, even if some irrelevant ones are included.

Random Retrieval performs better than BM-25 due to the Smaller Data size